## Facial Landmark Extraction - Setup & Test

### Context & Purpose
The UTKFace dataset provides cropped facial images with filename metadata (age, gender, ethnicity), but **does not ship with pre-annotated facial landmark coordinates**.

To analyze facial geometry—such as face width, face height, eye-to-eye distance, nose length, and mouth width—we extract 2D/3D facial landmarks programmatically using deep facial landmark detectors (such as **MediaPipe Face Mesh** and **OpenCV YuNet Landmark Detector**).

### Scope of Notebook 06
This notebook serves as a **pilot setup and validation test** on a reproducible sample of **100 images** to:
1. Verify model initialization and image loading pipelines.
2. Measure landmark detection success/failure rates on real dataset images.
3. Perform a sanity check on extracted normalized landmark coordinates `(x, y, z)` for key facial features (eyes, nose, mouth) prior to full-dataset batch extraction.

In [1]:
# Ensure working directory is set to project root
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Core imports with automatic fallback installation for opencv-python
import sys
import time
import urllib.request

try:
    import cv2
except ImportError:
    import subprocess
    print("Installing missing opencv-python package for active VS Code kernel...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "opencv-python"])
    import cv2
    print("opencv-python installed successfully!")

import numpy as np
import pandas as pd

# Re-initialize YuNet Face Landmark Detector
yunet_model_path = "models/face_detection_yunet_2023mar.onnx"
os.makedirs("models", exist_ok=True)
if not os.path.exists(yunet_model_path):
    yunet_url = "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx"
    print("Downloading YuNet ONNX model weights...")
    urllib.request.urlretrieve(yunet_url, yunet_model_path)

yunet_detector = cv2.FaceDetectorYN.create(
    model=yunet_model_path,
    config="",
    input_size=(200, 200),
    score_threshold=0.6,
    nms_threshold=0.3,
    top_k=5000
)
print("Detector initialized successfully.")


Detector initialized successfully.


In [2]:
# Initialize OpenCV YuNet Deep Facial Landmark Detector
# YuNet is a lightweight, high-precision neural face detector that extracts bounding boxes
# and 5 key facial landmarks: Right Eye, Left Eye, Nose Tip, Right Mouth Corner, Left Mouth Corner.

yunet_model_path = "models/face_detection_yunet_2023mar.onnx"

# Automatically download YuNet ONNX weights if not present locally
os.makedirs("models", exist_ok=True)
if not os.path.exists(yunet_model_path):
    yunet_url = "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx"
    print("Downloading YuNet landmark model weights...")
    urllib.request.urlretrieve(yunet_url, yunet_model_path)

# Instantiate FaceDetectorYN detector
yunet_detector = cv2.FaceDetectorYN.create(
    model=yunet_model_path,
    config="",
    input_size=(200, 200),
    score_threshold=0.5,
    nms_threshold=0.3,
    top_k=5000
)
print("YuNet landmark model initialized successfully.")

YuNet landmark model initialized successfully.


In [3]:
# Load encoded dataset CSV
encoded_csv_path = "data/processed/utkface_encoded.csv"
df = pd.read_csv(encoded_csv_path)

# Draw a reproducible random sample of 100 rows using random_state=42
sample_df = df.sample(n=100, random_state=42).reset_index(drop=True)

print(f"Loaded {len(df)} encoded records from '{encoded_csv_path}'.")
print(f"Sampled exactly {len(sample_df)} rows for landmark extraction test (random_state=42).")

Loaded 23705 encoded records from 'data/processed/utkface_encoded.csv'.
Sampled exactly 100 rows for landmark extraction test (random_state=42).


In [4]:
# Define landmark extraction function
# Normalized Coordinates Explanation:
# Landmark coordinates (x, y) are returned as normalized float values in range [0.0, 1.0],
# representing proportions relative to image width and height:
#   - x_pixel = x_norm * image_width
#   - y_pixel = y_norm * image_height
# Normalized coordinates enable scale-invariant distance measurements across varying image resolutions.

def extract_landmarks(image_path):
    """
    Reads an image from image_path and extracts facial landmarks.
    Returns a dictionary of normalized keypoint coordinates if detected, or None if no face is found.
    """
    if not os.path.exists(image_path):
        return None
        
    img = cv2.imread(image_path)
    if img is None:
        return None
        
    img_h, img_w, _ = img.shape
    yunet_detector.setInputSize((img_w, img_h))
    _, faces = yunet_detector.detect(img)
    
    if faces is not None and len(faces) > 0:
        face = faces[0]
        # Extract bounding box and 5 key landmarks (x, y in pixels)
        bbox = face[0:4] # [x, y, w, h]
        right_eye = (face[4] / img_w, face[5] / img_h)
        left_eye  = (face[6] / img_w, face[7] / img_h)
        nose_tip  = (face[8] / img_w, face[9] / img_h)
        right_mouth = (face[10] / img_w, face[11] / img_h)
        left_mouth  = (face[12] / img_w, face[13] / img_h)
        
        return {
            'bbox_norm': (bbox[0] / img_w, bbox[1] / img_h, bbox[2] / img_w, bbox[3] / img_h),
            'right_eye': right_eye,
            'left_eye': left_eye,
            'nose_tip': nose_tip,
            'right_mouth': right_mouth,
            'left_mouth': left_mouth,
            'image_size': (img_w, img_h)
        }
    return None

In [5]:
# Loop through sample_df, execute extract_landmarks on each image, and track detection metrics
successful_detections = 0
failed_detections = 0
failed_image_names = []
sample_successful_record = None

for idx, row in sample_df.iterrows():
    fname = row['image_name']
    fpath = row['filepath']
    
    landmarks = extract_landmarks(fpath)
    if landmarks is not None:
        successful_detections += 1
        if sample_successful_record is None:
            sample_successful_record = (fname, landmarks)
    else:
        failed_detections += 1
        failed_image_names.append(fname)

# Print detection summary statistics
print("--- LANDMARK EXTRACTION TEST RESULTS (n=100) ---")
print(f"Total images tested: {len(sample_df)}")
print(f"Successful detections: {successful_detections} / {len(sample_df)} ({(successful_detections/len(sample_df))*100:.1f}%)")
print(f"Failed detections:     {failed_detections} / {len(sample_df)} ({(failed_detections/len(sample_df))*100:.1f}%)")
print(f"Failed Image Names:    {failed_image_names if failed_image_names else 'None'}")

--- LANDMARK EXTRACTION TEST RESULTS (n=100) ---
Total images tested: 100
Successful detections: 100 / 100 (100.0%)
Failed detections:     0 / 100 (0.0%)
Failed Image Names:    None


In [6]:
# Sanity check landmark values for ONE successfully detected sample image
if sample_successful_record:
    sample_fname, lm_data = sample_successful_record
    img_w, img_h = lm_data['image_size']
    
    print("--- SANITY CHECK LANDMARK EXTRACTION ---")
    print(f"Image Name: '{sample_fname}'")
    print(f"Image Dimensions: {img_w} x {img_h} pixels")
    print("Extracted Normalized Landmark Coordinates (x_norm, y_norm):")
    print(f"  - Left Eye Outer Corner (Left Eye):   x={lm_data['left_eye'][0]:.5f}, y={lm_data['left_eye'][1]:.5f}")
    print(f"  - Right Eye Outer Corner (Right Eye): x={lm_data['right_eye'][0]:.5f}, y={lm_data['right_eye'][1]:.5f}")
    print(f"  - Nose Tip:                          x={lm_data['nose_tip'][0]:.5f}, y={lm_data['nose_tip'][1]:.5f}")
    print(f"  - Left Mouth Corner:                 x={lm_data['left_mouth'][0]:.5f}, y={lm_data['left_mouth'][1]:.5f}")
    print(f"  - Right Mouth Corner:                x={lm_data['right_mouth'][0]:.5f}, y={lm_data['right_mouth'][1]:.5f}")

--- SANITY CHECK LANDMARK EXTRACTION ---
Image Name: '21_0_2_20170116170741864.jpg.chip.jpg'
Image Dimensions: 200 x 200 pixels
Extracted Normalized Landmark Coordinates (x_norm, y_norm):
  - Left Eye Outer Corner (Left Eye):   x=0.68575, y=0.28068
  - Right Eye Outer Corner (Right Eye): x=0.29887, y=0.26959
  - Nose Tip:                          x=0.47779, y=0.47954
  - Left Mouth Corner:                 x=0.64955, y=0.67602
  - Right Mouth Corner:                x=0.31539, y=0.66602
